# Build and Deploy Production-Grade AI Agents on Amazon Bedrock AgentCore using Strands
## 00 — The Use Case, the Journey & Setup

### The use case: ElectroBazaar Customer Support

**ElectroBazaar** is an (imaginary) online electronics store in India. Every day its support team answers the same questions:

- *"Where is my order?"* — Priya from Bengaluru, waiting on a USB-C hub
- *"My keyboard is a week late!"* — Arjun from Hyderabad, frustrated
- *"I want to return my laptop"* — Kavya from Chennai, ₹72,990 on the line

We will build an AI agent that handles these conversations **end to end**: it looks up real customer data, follows company refund policy, remembers customers across visits, and runs on managed AWS infrastructure.

### The journey: prototype → production

A demo agent is easy to build. A production agent also needs *control*, *policy*, *memory*, and *hosting*. Each notebook adds one layer, and in the last one we package all the layers into an app and deploy it.

| # | Notebook | Production question it answers | Layer we add |
|---|---|---|---|
| 01 | Agent + Tools | Can the agent act on *real* business data? | Tools + instructions |
| 02 | Hooks | Can we *control* what it does — with code, not prompts? | Rate limiting + audit trail |
| 03 | Skills + Steering | Does it follow *company policy* every single time? | SOPs + policy enforcement |
| 04 | Memory | Does it *remember* the customer across sessions and restarts? | AgentCore Memory |
| 05 | Deploy | Can it run *at scale* with security, isolation and observability? | AgentCore Runtime |

```
 Customer ──► AgentCore Runtime ──► Strands Agent ──► Amazon Bedrock (LLM)
                 (microVM per          │  ├─ tools      (order / refund APIs)
                  session)             │  ├─ hooks      (rate limit, audit)
                                       │  ├─ steering   (refund policy, tone)
                                       │  └─ skills     (SOPs)
                                       └──► AgentCore Memory (short + long term)
```

**Strands Agents** = the open-source *agent harness* SDK (the loop, tools, hooks, memory).
**Amazon Bedrock AgentCore** = the managed platform that *runs* that harness in production.

### Prerequisites

| Requirement | Why |
|---|---|
| Python 3.10+ | Strands + AgentCore SDK |
| AWS credentials (`aws configure`) | Bedrock + AgentCore calls |
| Amazon Bedrock model access — **Claude Sonnet 4.6** | The LLM behind the agent |
| Node.js 20+ and the AgentCore CLI | Notebook 05 (deployment) |
| `uv` | Builds the deployment package |

Install the AgentCore CLI once (in a terminal):

```bash
sudo npm install -g @aws/agentcore --ignore-scripts
agentcore --version
```

> **Model choice is a config change, not a rewrite.** Everything reads `MODEL_ID` from the environment.
> Using AWS event credits? `export MODEL_ID=amazon.nova-pro-v1:0`. Want it faster? `export MODEL_ID=global.anthropic.claude-haiku-4-5-20251001-v1:0`.

In [6]:
# Run once (ideally inside a virtual environment)
!pip install -q -r requirements.txt

### Verify the environment

In [7]:
import shutil
import boto3
from importlib.metadata import version

for pkg in ["strands-agents", "bedrock-agentcore", "boto3"]:
    print(f"{pkg:<20}: {version(pkg)}")

identity = boto3.client("sts").get_caller_identity()
print(f"\nAWS account         : {identity['Account']}")
print(f"AWS identity        : {identity['Arn']}")
print(f"AWS region          : {boto3.session.Session().region_name}")

print(f"\nagentcore CLI       : {'found' if shutil.which('agentcore') else 'NOT FOUND (needed for 05)'}")
print(f"uv                  : {'found' if shutil.which('uv') else 'NOT FOUND (needed for 05)'}")

strands-agents      : 1.62.0
bedrock-agentcore   : 1.25.1
boto3               : 1.43.121

AWS account         : 829250931565
AWS identity        : arn:aws:iam::829250931565:user/CR-Yeshwanth
AWS region          : us-east-1

agentcore CLI       : found
uv                  : found


✅ Ready. Next: **01_agent_and_tools.ipynb** — give the agent access to ElectroBazaar's backend.